# Les 12 - Verkorting van het chatgeschiedenis met agent-schetsblad

Dit notebook laat zien hoe je context beheert in lange gesprekken met behulp van het Microsoft Agent Framework. Naarmate gesprekken langer worden, neemt het aantal tokens toe — uiteindelijk overschrijdt dit het contextvenster van het model. We lossen dit op met een **contextsamenvattingspatroon** en een **agent-schetsblad** voor blijvend geheugen.

## Wat je zult leren:
1. **Waarom contextbeheer belangrijk is**: Begrijpen van tokenlimieten en contextvensters
2. **Contextbewuste agenten**: Agenten bouwen die hun eigen gesprekscontext beheren
3. **Contextsamenvattingspatroon**: Hulpmiddelen gebruiken om de gespreksgeschiedenis in te korten
4. **Agent-schetsblad**: Blijvend geheugen dat contextreductie overleeft

## Vereisten:
- Azure OpenAI setup met geconfigureerde omgevingsvariabelen
- Begrip van basisagentconcepten uit vorige lessen


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Waarom Contextbeheer Belangrijk Is

Elke LLM heeft een beperkte **contextvenster** — het maximale aantal tokens dat het in één verzoek kan verwerken. Naarmate een gesprek met meerdere beurten vordert:

- **Token aantal groeit lineair** met elk gebruikersbericht en assistentantwoord.
- **Prompt tokens domineren de kosten** omdat de hele geschiedenis elke beurt opnieuw wordt verzonden.
- Uiteindelijk **overtreft het gesprek het contextvenster** en het model knipt af of geeft een foutmelding.

### Strategieën voor Contextbeheer

| Strategie | Hoe Het Werkt | Afweging |
|---|---|---|
| **Afkapping** | Oudste berichten verwijderen | Verliest vroege context |
| **Samenvatting** | Oudere berichten samenvatten | Enige details verloren, maar kernpunten behouden |
| **Krabbelschrift / Extern Geheugen** | Belangrijke feiten buiten het gesprek opslaan | Vereist tool-aanroepen, maar overleeft elke vermindering |

In dit notebook combineren we **samenvatting** met een **krabbelschrift-tool** zodat de agent continuïteit kan behouden, ook wanneer de gesprekshistorie wordt samengevat.


## Een contextbewuste agent maken


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Een Lang Gesprek Simuleren

Laten we een meerfasig gesprek doorlopen om te zien hoe context zich opstapelt. De agent moet belangrijke details (voorkeuren, budget, reisdata) over meerdere beurten onthouden en continuïteit tonen.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Merk op hoe de agent context uit eerdere beurten behoudt — hij weet van Japan, sushi, tempels, fotografie, het budget van $3000, solo reizen, en de tijdsperiode in april. In een kort gesprek werkt dit goed, maar naarmate het gesprek groeit, wordt de volledige geschiedenis duur om opnieuw te versturen.

Laten we het gesprek voortzetten met meer beurten om de opeenhoping van context te zien:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Samenvattingspatroon

Naarmate het gesprek groeit, kunnen we een **samenvattingshulpmiddel** gebruiken om de verzamelde context in een compact formaat samen te vatten. De agent roept dit hulpmiddel aan om belangrijke voorkeuren vast te leggen, zodat zelfs als oudere berichten worden verwijderd, de essentiële informatie behouden blijft.

Dit patroon is de bouwsteen voor meer geavanceerde geschiedenisreductie:
1. De agent identificeert belangrijke feiten uit het gesprek
2. Hij roept het samenvattingshulpmiddel aan om ze vast te leggen
3. Oudere berichten kunnen veilig worden verwijderd omdat de samenvatting vastlegt wat belangrijk is

Hieronder definiëren we een `summarize_preferences` hulpmiddel dat de agent kan aanroepen om een compacte samenvatting van wat hij heeft geleerd vast te leggen.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Samenvatting

In deze les heb je geleerd hoe je context beheert in langdurige agentgesprekken met behulp van het Microsoft Agent Framework:

### Belangrijke Concepten
- **Contextvensters zijn beperkt** — elke token in de gespreksgeschiedenis kost geld en telt mee voor de limiet.
- **Samenvattingstools** stellen de agent in staat om verzamelde context te condenseren tot compacte samenvattingen, wat tokengebruik vermindert terwijl essentiële informatie behouden blijft.
- **Agent kladblokken** bieden een blijvend extern geheugen dat elke conversiereductie overleeft.

### Wat Je Hebt Gebouwd
- Een **contextbewuste agent** die continuïteit bewaart over gesprekken met meerdere beurten
- Een **samenvattingstool** (`summarize_preferences`) die belangrijke gebruikersdetails in een compact formaat vastlegt
- Een **gesprek met meerdere beurten** die contextbehoud en wijzigingsverwerking demonstreert

### Toepassingen in de Praktijk
- **Klantenservicebots**: Onthoud voorkeuren gedurende lange ondersteuning sessies
- **Persoonlijke assistenten**: Volg lopende projecten zonder de context opnieuw uit te leggen
- **Onderwijzende tutors**: Houd de voortgang van studenten bij over veel interacties

### Volgende Stappen
- Implementeer een volledige kladbloktool met bestandsgebaseerde persistentie
- Voeg automatische geschiedenisafkapping toe na samenvatten
- Combineer met vectordatabases voor semantisch geheugenzoeken
- Bouw agents die gesprekken dagen later met volledige context kunnen hervatten


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
